In [7]:
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.pipeline import Pipeline

# ===============================================================
# 1. ЗАГРУЗКА
# ===============================================================
df = pd.read_csv("Предпочтение пользователей Iphone или android.csv", encoding="utf-8-sig")
df.columns = [
    "timestamp", "stability", "photo_freq", "blog", "pay_for",
    "status_tech", "vpn", "age", "budget", "customization",
    "ringtone", "resale", "system_setup", "change_freq",
    "os_support", "sphere", "gender", "instagram", "charges_per_day",
    "save_or_treat", "headphones", "apples", "target"
]

# ===============================================================
# 2. ОЧИСТКА
# ===============================================================
df["age"] = pd.to_numeric(df["age"], errors="coerce")
df = df[(df["age"] >= 14) & (df["age"] <= 80)]

def parse_budget(x):
    x = str(x).strip().replace(" ", "").replace(".", "").replace(",", ".")
    try:
        val = float(x)
    except ValueError:
        return np.nan
    return val * 1000 if val < 1000 else val

df["budget"] = df["budget"].apply(parse_budget)
df["change_freq"] = pd.to_numeric(df["change_freq"].astype(str).str.strip(), errors="coerce")
df["charges_per_day"] = pd.to_numeric(df["charges_per_day"], errors="coerce")
df["apples"] = pd.to_numeric(df["apples"], errors="coerce")
df = df.dropna(subset=["budget", "change_freq", "charges_per_day", "apples"])

df["target"] = df["target"].astype(str).str.strip().str.lower()
df = df[df["target"].isin(["айфон", "андроид"])].reset_index(drop=True)
df["target_bin"] = df["target"].map({"айфон": 1, "андроид": 0})

# ===============================================================
# 3. КОДИРОВАНИЕ
# ===============================================================
def split_multi(series):
    return series.fillna("").apply(
        lambda x: set(v.strip().lower() for v in re.split(r"[;,]", str(x)) if v.strip())
    )

tags = set()
for s in split_multi(df["pay_for"]):
    tags.update(s)

for tag in sorted(tags):
    df[f"pay_for_{tag}"] = split_multi(df["pay_for"]).apply(lambda s: int(tag in s))
df = df.drop(columns=["pay_for"])

cat_cols = [
    "stability", "photo_freq", "blog", "status_tech", "vpn",
    "ringtone", "resale", "system_setup", "os_support",
    "sphere", "gender", "instagram", "save_or_treat", "headphones"
]
encoders = {}
for col in cat_cols:
    df[col] = df[col].astype(str).str.strip().str.lower()
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    encoders[col] = le

feature_cols = [c for c in df.columns if c not in ["timestamp", "target", "target_bin"]]
X = df[feature_cols].astype(float)
y = df["target_bin"].astype(int)

# ===============================================================
# 4. ОБУЧЕНИЕ
# ===============================================================
X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X, y, df.index, test_size=0.25, random_state=42, stratify=y
)

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier())
])
param_grid = {
    "knn__n_neighbors": range(1, 16),
    "knn__weights": ["uniform", "distance"],
    "knn__metric": ["euclidean", "manhattan"]
}
grid = GridSearchCV(pipeline, param_grid, cv=5, scoring="accuracy", n_jobs=-1)
grid.fit(X_train, y_train)
model = grid.best_estimator_

print("=" * 70)
print("ОБУЧЕНИЕ ЗАВЕРШЕНО")
print("=" * 70)
print(f"Лучшие параметры: {grid.best_params_}")
print(f"Точность на кросс-валидации: {grid.best_score_:.1%}")
print(f"Обучающая выборка: {len(X_train)} чел. | Тестовая: {len(X_test)} чел.")
print()

# ===============================================================
# 5. ПОНЯТНЫЙ ВЫВОД ПО КАЖДОМУ ТЕСТУ
# ===============================================================
pred_all = model.predict(X)
pred_test = model.predict(X_test)

# --- 5.1. Прогноз по всей выборке ---
print("=" * 70)
print("ПРЕДСКАЗАНИЯ ПО КАЖДОМУ РЕСПОНДЕНТУ (вся выборка)")
print("=" * 70)
print(f"{'№':>3} | {'Возраст':>7} | {'Бюджет':>9} | {'Сфера':<12} | "
      f"{'Предсказано':<11} | {'Реально':<10} | {'Итог'}")
print("-" * 70)

correct_total = 0
for i, (_, row) in enumerate(df.iterrows(), start=1):
    real = "Айфон" if row["target_bin"] == 1 else "Андроид"
    pred = "Айфон" if pred_all[i - 1] == 1 else "Андроид"
    ok = "+" if real == pred else "-"
    if real == pred:
        correct_total += 1
    sphere = str(row["sphere"])[:12]
    print(f"{i:>3} | {int(row['age']):>7} | {int(row['budget']):>9} | "
          f"{sphere:<12} | {pred:<11} | {real:<10} | {ok}")

print("-" * 70)
print(f"Итого угадано: {correct_total} из {len(df)} ({correct_total / len(df):.1%})")
print()

# --- 5.2. Прогноз только по тестовой выборке ---
print("=" * 70)
print("ПРЕДСКАЗАНИЯ ПО ТЕСТОВОЙ ВЫБОРКЕ (25% данных)")
print("=" * 70)
print(f"{'№':>3} | {'Возраст':>7} | {'Бюджет':>9} | {'Сфера':<12} | "
      f"{'Предсказано':<11} | {'Реально':<10} | {'Итог'}")
print("-" * 70)

correct_test = 0
for j, (idx, pred) in enumerate(zip(idx_test, pred_test), start=1):
    row = df.loc[idx]
    real = "Айфон" if row["target_bin"] == 1 else "Андроид"
    pred_lbl = "Айфон" if pred == 1 else "Андроид"
    ok = "+" if real == pred_lbl else "-"
    if real == pred_lbl:
        correct_test += 1
    sphere = str(row["sphere"])[:12]
    print(f"{j:>3} | {int(row['age']):>7} | {int(row['budget']):>9} | "
          f"{sphere:<12} | {pred_lbl:<11} | {real:<10} | {ok}")

print("-" * 70)
print(f"Итого угадано: {correct_test} из {len(X_test)} ({correct_test / len(X_test):.1%})")
print()

# ===============================================================
# 6. ОБЩИЙ РЕЗУЛЬТАТ
# ===============================================================
cm = confusion_matrix(y, pred_all)
tp, fn, fp, tn = cm[1, 1], cm[1, 0], cm[0, 1], cm[0, 0]

print("=" * 70)
print("ИТОГОВЫЙ РЕЗУЛЬТАТ")
print("=" * 70)
print(f"Всего опрошенных:        {len(df)}")
print(f"Айфонов в данных:        {(y == 1).sum()}")
print(f"Андроидов в данных:      {(y == 0).sum()}")
print()
print(f"Общая точность модели:   {accuracy_score(y, pred_all):.1%}")
print(f"Точность на тесте:       {accuracy_score(y_test, pred_test):.1%}")
print()
print("Матрица ошибок (вся выборка):")
print(f"  Правильно угадан Айфон:   {tp}")
print(f"  Айфон принят за Андроид:  {fn}")
print(f"  Андроид принят за Айфон:  {fp}")
print(f"  Правильно угадан Андроид: {tn}")
print()
print("Качество по классам:")
print(f"  Айфон  — точность {tp / (tp + fp):.1%}, полнота {tp / (tp + fn):.1%}")
print(f"  Андроид — точность {tn / (tn + fn):.1%}, полнота {tn / (tn + fp):.1%}")
print()


ОБУЧЕНИЕ ЗАВЕРШЕНО
Лучшие параметры: {'knn__metric': 'euclidean', 'knn__n_neighbors': 4, 'knn__weights': 'distance'}
Точность на кросс-валидации: 80.0%
Обучающая выборка: 30 чел. | Тестовая: 10 чел.

ПРЕДСКАЗАНИЯ ПО КАЖДОМУ РЕСПОНДЕНТУ (вся выборка)
  № | Возраст |    Бюджет | Сфера        | Предсказано | Реально    | Итог
----------------------------------------------------------------------
  1 |      21 |    800000 | 0            | Андроид     | Андроид    | +
  2 |      21 |    900000 | 14           | Айфон       | Айфон      | +
  3 |      22 |    600000 | 9            | Айфон       | Айфон      | +
  4 |      22 |    500000 | 17           | Айфон       | Айфон      | +
  5 |      22 |    300000 | 18           | Андроид     | Андроид    | +
  6 |      22 |    700000 | 0            | Айфон       | Айфон      | +
  7 |      30 |    500000 | 10           | Айфон       | Айфон      | +
  8 |      21 |    700000 | 4            | Андроид     | Айфон      | -
  9 |      22 |    200000 | 